In [1]:
"""Create FAKE daily health counts for learning the weather-only CNN workflow.

No real health records are read. The weather effect is deliberately invented;
these data cannot establish a real weather-health relationship.

Set WEATHER_FILE to your DAILY weather NetCDF for matching dates and an
invented weather signal. Set it to None for calendar-only demonstration data.
Output columns match one_branch_cnn.py: date,visits,baseline_visits.
Dependencies: numpy pandas; xarray and netCDF4 when reading weather.
"""
from pathlib import Path
import numpy as np
import pandas as pd

# Use the same DAILY weather file as in one_branch_cnn.py.
# Units: gph in hpa Dimensions as in that script.
WEATHER_FILE = Path(r"D:\ERA 5 Data\ERA5_Camden\daily_weather.nc")

# To try this generator without a weather file, change the line above to:
#WEATHER_FILE = None

OUTPUT_FILE = Path(r"D:\ERA 5 Data\ERA5_Camden\daily_health_sample.csv")
SEED = 20
START_DATE = "2004-01-01"  # used only when WEATHER_FILE is None
END_DATE = "2024-12-31"



def generate_sample(weather_file=WEATHER_FILE, seed=SEED):
    rng = np.random.default_rng(seed)

    if weather_file is not None:
        import xarray as xr

        with xr.open_dataset(weather_file, chunks=None) as ds:
            if "gph" not in ds:
                raise ValueError("Dataset must contain 'gph'.")

            if set(ds["gph"].dims) != {
                "valid_time", "latitude", "longitude"
            }:
                raise ValueError(
                    "gph needs valid_time, latitude, longitude dimensions."
                )

            ds = ds.sortby("valid_time")
            dates = pd.DatetimeIndex(ds.valid_time.values)

            if (
                dates.has_duplicates
                or dates.hasnans
                or not dates.equals(dates.normalize())
            ):
                raise ValueError(
                    "Weather must have one midnight date label per daily record."
                )

            # Spatial mean of geopotential height
            gph_mean = ds["gph"].mean(
                ("latitude", "longitude"),
                skipna=False
            ).values

        if len(dates) == 0 or not np.isfinite(gph_mean).all():
            raise ValueError(
                "Weather must be nonempty and contain finite values."
            )

    else:
        dates = pd.date_range(START_DATE, END_DATE, freq="D")

        # Synthetic geopotential height for demonstration only
        gph_mean = 5600 + rng.normal(0, 80, len(dates))

    # Health outcomes occur the NEXT calendar day
    health_dates = dates + pd.Timedelta(days=1)

    doy = health_dates.dayofyear.to_numpy()
    weekend = (
        health_dates.dayofweek.to_numpy() >= 5
    ).astype(float)

    # Fabricated baseline visits
    baseline = (
        100
        + 8 * np.cos(2 * np.pi * (doy - 15) / 365.25)
        - 6 * weekend
    )

    # Synthetic relationship between GPH and visits
    # Arbitrary coefficient for testing, NOT a scientific estimate
    gph_anomaly = gph_mean - 5600

    weather_effect_pct = 0.05 * gph_anomaly

    other_variation_pct = rng.normal(0, 4, len(dates))

    expected_visits = baseline * (
        1 + (weather_effect_pct + other_variation_pct) / 100
    )

    expected_visits = np.maximum(expected_visits, 1)

    visits = rng.poisson(expected_visits)

    return pd.DataFrame({
        "date": health_dates.strftime("%Y-%m-%d"),
        "visits": visits.astype(int),
        "baseline_visits": baseline.round(3),
    })



def main():
    if WEATHER_FILE is not None and not WEATHER_FILE.exists():
        raise FileNotFoundError(
            f"Daily weather file not found: {WEATHER_FILE}. "
            "Edit WEATHER_FILE to your path, or set it to None for calendar-only sample data."
        )
    sample = generate_sample()
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(OUTPUT_FILE, index=False)
    excess = 100 * (sample.visits - sample.baseline_visits) / sample.baseline_visits
    print("SYNTHETIC DATA ONLY: invented counts and weather effects.")
    print(f"Saved {len(sample)} rows to {OUTPUT_FILE.resolve()}")
    print(sample.head().to_string(index=False))
    print(f"Observed synthetic excess range: {excess.min():.1f}% to {excess.max():.1f}%")
    print("In one_branch_cnn.py, set HEALTH_FILE to this CSV's absolute path.")
    if WEATHER_FILE is None:
        print("Calendar-only mode: these counts were not generated from your actual weather.")


In [2]:
print(OUTPUT_FILE.exists())

True


In [7]:
sample = generate_sample(weather_file=None)

OUTPUT_FILE = Path(
    r"D:\ERA 5 Data\ERA5_Camden\daily_health_sample.csv"
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(OUTPUT_FILE, index=False)

print("Saved to:", OUTPUT_FILE)
print("File exists:", OUTPUT_FILE.exists())
print(sample.head())

Saved to: D:\ERA 5 Data\ERA5_Camden\daily_health_sample.csv
File exists: True
         date  visits  baseline_visits
0  2004-01-02     111          107.801
1  2004-01-03      95          101.830
2  2004-01-04     101          101.857
3  2004-01-05      95          107.882
4  2004-01-06     114          107.904
